In [ ]:
import os
from dotenv import load_dotenv

# Cargamos el .env de 01_rag_api (un nivel arriba de notebooks/)
load_dotenv(os.path.join("..", ".env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")
ASSISTANT_NAME = os.getenv("ASSISTANT_NAME", "Asistente")

QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")
QDRANT_COLLECTION_NAME = os.getenv("QDRANT_COLLECTION_NAME")

print("Modelo LLM:", OPENAI_MODEL)
print("Modelo de embeddings:", OPENAI_EMBEDDING_MODEL)
print("Coleccion de Qdrant:", QDRANT_COLLECTION_NAME)

## 1. Conectarnos a Qdrant (base de datos vectorial ya existente)

Creamos un `QdrantClient` que apunta a la coleccion existente, y un `QdrantVectorStore` de LangChain que sabe como buscar sobre esa coleccion usando embeddings de OpenAI.

Esto es equivalente a lo que hace `QdrantIntegration.__init__` en `src/integrations/impl/qdrant_integration.py`.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

# Cliente de Qdrant apuntando a la coleccion ya existente
client = QdrantClient(
    url=QDRANT_URL,
    api_key=QDRANT_API_KEY,
    timeout=60,
)

# Modelo de embeddings de OpenAI (el mismo usado al indexar los documentos)
embeddings = OpenAIEmbeddings(api_key=OPENAI_API_KEY, model=OPENAI_EMBEDDING_MODEL)

# VectorStore de LangChain: envuelve el cliente + la coleccion + el modelo de embeddings
vector_store = QdrantVectorStore(
    client=client,
    collection_name=QDRANT_COLLECTION_NAME,
    embedding=embeddings,
)

info = client.get_collection(QDRANT_COLLECTION_NAME)
print("Conectado a la coleccion:", QDRANT_COLLECTION_NAME)
print("Puntos indexados:", info.points_count)

In [ ]:
# --- Paso 1: Generar el embedding de la pregunta ---
# Convertimos la pregunta del usuario en un vector numerico (embedding),
# usando el mismo modelo de embeddings con el que se indexaron los documentos.
# Equivalente a OpenAIEmbeddingIntegration.generate_embedding
# (src/integrations/impl/openai_embedding_integration.py)

question = "que programas tiene?"

vector = embeddings.embed_query(question)

print("Dimension del embedding:", len(vector))
print("Primeros 5 valores:", vector[:5])

## 3. Paso 2 - Buscar documentos similares en Qdrant (retrieval)

Con el embedding de la pregunta, buscamos en Qdrant los `k` documentos mas parecidos (similitud de coseno). Cada resultado viene con su `score` de similitud.

Esto es lo que hace `QdrantIntegration.search` en `src/integrations/impl/qdrant_integration.py`.

In [ ]:
RETRIEVAL_LIMIT = 5

# Buscamos los documentos mas similares al vector de la pregunta
docs_with_scores = vector_store.similarity_search_with_score_by_vector(
    vector, k=RETRIEVAL_LIMIT
)

for doc, score in docs_with_scores:
    print(f"score={score:.4f} | source={doc.metadata.get('source_file', '')}")
    print(doc.page_content[:200], "...\n")

In [ ]:
# Cada resultado trae (documento, score); solo nos interesa el texto (page_content)
formatted_context = "\n".join([f"- {doc.page_content}" for doc, _score in docs_with_scores])

print(formatted_context)

## 4. Paso 3 - Construir el prompt (instrucciones + contexto + pregunta)

El prompt tiene 3 partes, igual que en `src/integrations/impl/prompt.py` y `OpenAILlmIntegration`:

1. **System prompt** (`INSTRUCTIONS`): define la personalidad/reglas del asistente.
2. **Historial** (lo omitimos aqui, ya que este notebook se enfoca solo en el RAG).
3. **Human prompt** (`USER_PROMPT`): inyecta el contexto recuperado y la pregunta del usuario.

Usamos `ChatPromptTemplate` de LangChain para armar esta plantilla.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Mismas plantillas que src/integrations/impl/prompt.py
INSTRUCTIONS = (
    "Eres un asistente útil y amigable. Responde siempre en español."
    "Tu nombre es {assistant_name}."
    "Sé conciso en tus respuestas."
)

USER_PROMPT = (
    "Contexto:\n"
    "{context}\n\n"
    "Pregunta: {question}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", INSTRUCTIONS),
        ("human", USER_PROMPT),
    ]
)

# Vemos como queda el prompt final ya "resuelto" con nuestros valores
mensajes = prompt.format_messages(
    assistant_name=ASSISTANT_NAME,
    question=question,
    context=formatted_context,
)

for m in mensajes:
    print(f"--- {m.type} ---")
    print(m.content, "\n")

## 5. Paso 4 - Invocar al LLM

Finalmente, enviamos el prompt al modelo de OpenAI (`ChatOpenAI`) y obtenemos la respuesta generada. Encadenamos `prompt | llm` usando LCEL (LangChain Expression Language), igual que en `OpenAILlmIntegration.__init__`.

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(api_key=OPENAI_API_KEY, model=OPENAI_MODEL)

# Encadenamos: el prompt formateado se envia directo al LLM
chain = prompt | llm

response = chain.invoke(
    {
        "assistant_name": ASSISTANT_NAME,
        "question": question,
        "context": formatted_context,
    }
)

usage = response.usage_metadata or {}

print("Pregunta del usuario:")
print(question)
print("\nRespuesta del asistente:\n")
print(response.content)
print("\nTokens de entrada:", usage.get("input_tokens", 0))
print("Tokens de salida:", usage.get("output_tokens", 0))